# Filter hyperparameters & output size

_Deep Learning_

**Filter size, stride, and padding decide how big the output map turns out.**

Three settings control a convolution: the filter size, the stride, and the padding.

     Filter size $F$ is how wide the filter is. Stride $S$ is how many pixels it jumps each slide. Padding $P$ is extra zeros added around the edge.

---

This notebook works through the output-size formula one setting at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — NumPy

The output size of a convolution follows one formula: `(n + 2P - F) / S + 1`, rounded down. We build it in two steps: (1) write the formula as a function and try the three classic cases, (2) check that "same" padding really preserves size for any odd filter.

### Step 1 — The output-size formula

For an input of width `n`, a filter of width `F`, padding `P`, and stride `S`, the output width is `(n + 2P - F) // S + 1`. Padding adds `P` zeros on *each* side (hence `2P`), and the stride divides because the filter jumps `S` pixels per step. We try three classic settings on a 28x28 input: plain "valid" (no padding), "same" padding (size preserved), and stride 2 (downsampling).

In [ ]:
import numpy as np

def out_size(n, f, p, s):
    # n=input size, f=filter, p=padding, s=stride
    return (n + 2 * p - f) // s + 1

# 28x28 input, 5x5 filter
print("valid (p=0, s=1):", out_size(28, 5, 0, 1))   # 24
print("same  (p=2, s=1):", out_size(28, 5, 2, 1))   # 28 (size preserved)
print("stride 2 (p=0):  ", out_size(28, 5, 0, 2))   # 12 (downsampled)

### Step 2 — Why "same" padding uses P = (F-1)/2

To keep the output the same width as the input (with stride 1), you pad by `(F-1)/2` on each side. That value is a whole number only for odd filter sizes, which is why conv filters are almost always odd (3, 5, 7). We loop over several odd filters and confirm the output stays 28 every time.

In [ ]:
# verify "same" padding keeps the size for any odd filter
for f in [1, 3, 5, 7]:
    p = (f - 1) // 2
    out = out_size(28, f, p, 1)
    print("f=%d same padding p=%d -> out=%d" % (f, p, out))

## Visualize the data & results

_How do padding and stride change a conv's output size over a real 8x8 digit image?_

### Step 1 — Compute output sizes for several configs on a real image

We take the real 8x8 digit image side (`n = 8`) and run the same formula across five filter/padding/stride combinations. This shows concretely how each knob moves the result: padding grows the output, larger filters shrink it, and stride halves it.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
n = digits.images[0].shape[0]            # real image side = 8

def out_size(n, f, p, s):
    return (n + 2 * p - f) // s + 1      # conv output-size formula

configs = [("3x3 valid p0 s1", 3, 0, 1),
           ("3x3 same p1 s1", 3, 1, 1),
           ("3x3 stride2", 3, 0, 2),
           ("5x5 valid p0", 5, 0, 1),
           ("5x5 same p2", 5, 2, 1)]

labels = [name for name, f, p, s in configs]
values = [out_size(n, f, p, s) for name, f, p, s in configs]
for name, v in zip(labels, values):
    print("%-16s -> %d" % (name, v))

### Step 2 — Plot the output sizes

We bar-chart the five output sizes side by side. The "same" configs return to 8, the valid configs shrink below 8, and the stride-2 config drops the furthest — a quick visual of how the three hyperparameters trade off.

In [ ]:
colors = ["#4ea1ff", "#7ee787", "#ffb454", "#ff7b72", "#c89bff"]

plt.bar(labels, values, color=colors)
plt.title("Conv output size over a real 8x8 digit (input=8)")
plt.xticks(rotation=20)
plt.show()